# 20 RNN 与 LSTM

依赖安装说明：`pip install numpy matplotlib scikit-learn torch`

RNN 用来处理序列数据。它按时间一步步读入输入，并维护隐藏状态。LSTM 是 RNN 的改进版，更擅长保留长期信息。


## 1. 数学逻辑

普通 RNN：

$$h_t = \tanh(W_xx_t + W_hh_{t-1}+b)$$

输出可以由最后一个隐藏状态得到：

$$\hat y = W_oh_T+b_o$$

LSTM 增加门控机制：

$$f_t=\sigma(W_f[x_t,h_{t-1}]+b_f)$$

$$i_t=\sigma(W_i[x_t,h_{t-1}]+b_i)$$

$$o_t=\sigma(W_o[x_t,h_{t-1}]+b_o)$$

门决定忘掉什么、写入什么、输出什么。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# 合成序列任务：长度为 12 的 0/1 序列，标签为前半段 1 的数量是否超过后半段
n, seq_len = 500, 12
X = np.random.randint(0, 2, size=(n, seq_len, 1)).astype('float32')
y = (X[:, :6, 0].sum(axis=1) > X[:, 6:, 0].sum(axis=1)).astype('int64')
print('X shape:', X.shape, 'y mean:', y.mean())


In [ ]:
# 从零演示：RNN 隐藏状态如何逐步更新，不做完整训练
hidden_size = 4
Wx = np.random.normal(scale=0.5, size=(1, hidden_size))
Wh = np.random.normal(scale=0.5, size=(hidden_size, hidden_size))
b = np.zeros(hidden_size)

h = np.zeros(hidden_size)
example = X[0]
for t, xt in enumerate(example):
    h = np.tanh(xt @ Wx + h @ Wh + b)
    print(f't={t:2d} | x={int(xt[0])} | h={np.round(h, 3)}')


In [ ]:
# PyTorch 实战：LSTM 做序列分类
import torch
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y)
Xtr = torch.tensor(X_train, dtype=torch.float32)
ytr = torch.tensor(y_train, dtype=torch.long)
Xte = torch.tensor(X_test, dtype=torch.float32)

class LSTMClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=12, batch_first=True)
        self.head = nn.Linear(12, 2)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :])

model = LSTMClassifier()
optimizer = torch.optim.Adam(model.parameters(), lr=0.02)
loss_fn = nn.CrossEntropyLoss()

for step in range(200):
    logits = model(Xtr)
    loss = loss_fn(logits, ytr)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

with torch.no_grad():
    pred = model(Xte).argmax(dim=1).numpy()
print('LSTM accuracy:', round(accuracy_score(y_test, pred), 3))


## 2. 常见误区

- RNN 按顺序处理，训练通常比 CNN/Transformer 难并且慢。
- 普通 RNN 容易梯度消失，LSTM/GRU 能缓解但不是万能。
- 序列最后一步不一定包含所有信息，任务不同可能需要 attention 或 pooling。

## 3. 小实验

- 改 `seq_len`，观察长序列难度。
- 把 LSTM 换成 `nn.GRU`。
- 增加 hidden size，看效果和速度。
